In [ ]:
!git clone https://github.com/ParticleMedia/RAGTruth.git

Cloning into 'RAGTruth'...
remote: Enumerating objects: 61, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (25/25), done.
remote: Total 61 (delta 10), reused 22 (delta 7), pack-reused 28 (from 1)
Receiving objects: 100% (61/61), 9.29 MiB | 11.73 MiB/s, done.
Resolving deltas: 100% (20/20), done.


In [ ]:
import os

DATA_DIR = "/content/RAGTruth/dataset"

print("Dataset folder exists:", os.path.exists(DATA_DIR))
print("Files:", os.listdir(DATA_DIR))

Dataset folder exists: True
Files: ['source_info.jsonl', 'response.jsonl']


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/rag-eval-metrics"
os.makedirs(PROJECT_DIR, exist_ok=True)

print("Project folder ready:", PROJECT_DIR)

Project folder ready: /content/drive/MyDrive/rag-eval-metrics


Load And Inspect the RAGTruth

In [ ]:
import pandas as pd

DATA_DIR = "/content/RAGTruth/dataset"

responses = pd.read_json(
    f"{DATA_DIR}/response.jsonl",
    lines=True
)

sources = pd.read_json(
    f"{DATA_DIR}/source_info.jsonl",
    lines=True
)

print("Responses shape:", responses.shape)
print("Sources shape:", sources.shape)

print("\nResponse columns:")
print(responses.columns.tolist())

print("\nSource columns:")
print(sources.columns.tolist())

Responses shape: (17790, 8)
Sources shape: (2965, 5)

Response columns:
['id', 'source_id', 'model', 'temperature', 'labels', 'split', 'quality', 'response']

Source columns:
['source_id', 'task_type', 'source', 'source_info', 'prompt']


In [ ]:
#Merge, sample, and prepare the evaluation dataframe

import pandas as pd
import ast
import os

# 1. Merge each response with its source context and task type
merged = responses.merge(
    sources[["source_id", "task_type", "source_info"]],
    on="source_id",
    how="left"
)

print("Merged dataset shape:", merged.shape)
print("\nTask-type distribution:")
print(merged["task_type"].value_counts())

# 2. Rebuild the approximately 1,800-row stratified sample
# Keep the same seed and proportional sampling procedure as before.
sample_size = 1800

sampled = (
    merged.groupby("task_type", group_keys=False)
    .apply(
        lambda group: group.sample(
            n=round(len(group) / len(merged) * sample_size),
            random_state=42
        )
    )
    .reset_index(drop=True)
)

print("\nStratified sample shape:", sampled.shape)
print("\nSample task-type distribution:")
print(sampled["task_type"].value_counts())

# 3. Exclude responses with generation-quality issues
exclude_quality = ["incorrect_refusal", "truncated"]

excluded_mask = sampled["quality"].isin(exclude_quality)

print("\nExcluded rows by quality:")
print(sampled.loc[excluded_mask, "quality"].value_counts())
print("Total excluded:", excluded_mask.sum())

eval_df = sampled.loc[~excluded_mask].copy()

# 4. Ensure labels are Python lists after any CSV round-trip
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value) or value == "":
        return []
    return ast.literal_eval(value)

eval_df["labels"] = eval_df["labels"].apply(parse_labels)

# 5. Create response-level hallucination ground truth
eval_df["has_hallucination"] = eval_df["labels"].apply(
    lambda labels: len(labels) > 0
)

print("\nCanonical evaluation dataframe shape:", eval_df.shape)

print("\nHallucination label distribution:")
print(eval_df["has_hallucination"].value_counts())

print(
    "Hallucinated response proportion:",
    round(eval_df["has_hallucination"].mean(), 3)
)

print("\nHallucination distribution by task type:")
print(
    eval_df.groupby("task_type")["has_hallucination"]
    .value_counts()
    .unstack(fill_value=0)
)

# 6. Save both datasets to Google Drive
sample_path = os.path.join(
    PROJECT_DIR, "ragtruth_stratified_sample.csv"
)
eval_path = os.path.join(
    PROJECT_DIR, "ragtruth_eval_dataframe.csv"
)

sampled.to_csv(sample_path, index=False)
eval_df.to_csv(eval_path, index=False)

print("\nSaved files:")
print(sample_path)
print(eval_path)

Merged dataset shape: (17790, 10)

Task-type distribution:
task_type
Data2txt    6198
QA          5934
Summary     5658
Name: count, dtype: int64

Stratified sample shape: (1799, 10)

Sample task-type distribution:
task_type
Data2txt    627
QA          600
Summary     572
Name: count, dtype: int64

Excluded rows by quality:
quality
incorrect_refusal    16
truncated             2
Name: count, dtype: int64
Total excluded: 18

Canonical evaluation dataframe shape: (1781, 11)

Hallucination label distribution:
has_hallucination
False    1017
True      764
Name: count, dtype: int64
Hallucinated response proportion: 0.429

Hallucination distribution by task type:
has_hallucination  False  True 
task_type                      
Data2txt             220    407
QA                   405    178
Summary              392    179


/tmp/ipykernel_744/2412348017.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(



Saved files:
/content/drive/MyDrive/rag-eval-metrics/ragtruth_stratified_sample.csv
/content/drive/MyDrive/rag-eval-metrics/ragtruth_eval_dataframe.csv
